# TFT Set 17 — Data Processing & Feature Engineering
Runs once (or when new match data arrives). Outputs `data/tft_clean.csv`.

In [17]:
import json
import pandas as pd
import os
import requests as _requests

def load_json(path):
    if not os.path.exists(path):
        print(f"  [skipped] {path} not found")
        return []
    with open(path) as f:
        data = json.load(f)
    print(f"  Loaded {len(data):,} matches from {path}")
    return data

my_matches = load_json("data/matches.json")
chall_matches = load_json("data/challenger_matches.json")

all_matches = my_matches + chall_matches
print(f"\nTotal: {len(all_matches):,} matches combined")

  Loaded 100 matches from data/matches.json
  Loaded 1,523 matches from data/challenger_matches.json

Total: 1,623 matches combined


## Extract raw stats

In [18]:
STAGE_SIZES = [4, 6, 7, 7, 5, 5, 5, 5]
_STAGE_STARTS, _total = [], 0
for s in STAGE_SIZES:
    _STAGE_STARTS.append(_total + 1)
    _total += s

def round_to_stage(r):
    for stage, (start, size) in enumerate(zip(_STAGE_STARTS, STAGE_SIZES), 1):
        if r <= start + size - 1:
            return f"{stage}-{r - start + 1}"
    rem = r - _total
    return f"{9 + (rem-1)//5}-{(rem-1)%5+1}"

def extract_stats(matches):
    rows = []
    for match in matches:
        if match["info"].get("tft_set_number") != 17:
            continue
        puuid = match["_my_puuid"]
        info  = match["info"]
        me    = next((p for p in info["participants"] if p["puuid"] == puuid), None)
        if me is None:
            continue
        lr = me["last_round"]
        rows.append({
            "match_id":          match["metadata"]["match_id"],
            "date":              pd.to_datetime(info["game_datetime"], unit="ms"),
            "player_name":       match.get("_player_name", "me"),
            "is_me":             match.get("_is_me", True),
            "placement":         me["placement"],
            "level":             me["level"],
            "last_round":        lr,
            "stage_round":       round_to_stage(lr),
            "gold_left":         me["gold_left"],
            "damage_to_players": me.get("total_damage_to_players", 0),
            "game_length_mins":  round(info.get("game_length", 0) / 60, 1),
            "augments":          [a.replace("TFT_Augment_", "") for a in me.get("augments", [])],
            "traits":            me.get("traits", []),
            "units":             me.get("units", []),
        })
    return pd.DataFrame(rows)

df = extract_stats(all_matches)
print(f"{len(df):,} rows  |  {df['is_me'].sum()} my games  |  {(~df['is_me']).sum()} challenger games")
df[["placement", "level", "last_round", "stage_round", "gold_left",
    "damage_to_players", "game_length_mins"]].head()

1,596 rows  |  75 my games  |  1521 challenger games


,placement,level,last_round,stage_round,gold_left,damage_to_players,game_length_mins
0,4,9,34,6-5,0,105,34.1
1,6,9,31,6-2,7,110,34.9
2,4,9,35,7-1,1,143,38.1
3,5,9,31,6-2,1,94,37.0
4,5,8,31,6-2,19,83,40.0


## Parse units — cost tiers, itemization, star levels

In [19]:
RARITY_TO_COST = {0: 1, 1: 2, 2: 3, 4: 4, 6: 5}

def parse_units(units_list):
    by_cost  = {c: [] for c in range(1, 6)}
    itemized = []
    for u in units_list:
        if not u["character_id"].startswith("TFT17_"):
            continue
        cost = RARITY_TO_COST.get(u["rarity"])
        if cost is None:
            continue
        name  = u["character_id"].split("_", 1)[-1]
        tier  = u.get("tier", 1)
        items = [i.replace("TFT_Item_", "").replace("TFT4_Item_", "")
                 for i in u.get("itemNames", [])]
        unit_info = {"unit": name, "tier": tier, "items": items}
        by_cost[cost].append(unit_info)
        if items:
            itemized.append({**unit_info, "cost": cost, "n_items": len(items)})
    itemized.sort(key=lambda x: x["n_items"], reverse=True)
    return by_cost, itemized

parsed = df["units"].apply(parse_units)
df["_parsed_by_cost"] = parsed.apply(lambda x: x[0])
df["itemized_units"]  = parsed.apply(lambda x: x[1])

for cost in range(1, 6):
    df[f"n_{cost}cost"] = df["_parsed_by_cost"].apply(lambda d: len(d[cost]))

df.drop(columns=["_parsed_by_cost"], inplace=True)
df[["placement", "n_1cost", "n_2cost", "n_3cost", "n_4cost", "n_5cost"]].head()

,placement,n_1cost,n_2cost,n_3cost,n_4cost,n_5cost
0,4,3,1,2,2,1
1,6,1,2,1,3,2
2,4,3,1,2,3,0
3,5,0,2,1,2,4
4,5,4,2,1,0,1


## CommunityDragon — unit roles

In [20]:
UNIT_DATA_PATH = "data/tft_units.json"

if not os.path.exists(UNIT_DATA_PATH):
    print("Fetching Set 17 unit data from CommunityDragon...")
    raw = _requests.get("https://raw.communitydragon.org/latest/cdragon/tft/en_us.json",
                        timeout=60).json()
    set17 = next(s for s in raw["setData"] if s["number"] == 17)
    units = [
        {"apiName": u["apiName"], "name": u["name"], "cost": u["cost"],
         "role": u.get("role") or "", "traits": u.get("traits", [])}
        for u in set17["champions"]
        if u["apiName"].startswith("TFT17_") and u.get("cost", 99) <= 5
    ]
    with open(UNIT_DATA_PATH, "w") as f:
        json.dump(units, f, indent=2)
    print(f"Saved {len(units)} units → {UNIT_DATA_PATH}")
else:
    with open(UNIT_DATA_PATH) as f:
        units = json.load(f)
    print(f"Loaded {len(units)} Set 17 units from cache")

UNIT_LOOKUP = {u["apiName"].replace("TFT17_", ""): u for u in units}
print(f"UNIT_LOOKUP has {len(UNIT_LOOKUP)} entries")

Loaded 65 Set 17 units from cache
UNIT_LOOKUP has 65 entries


In [21]:
ROLE_OVERRIDES = {"MissFortune": "ADCarry"}

def get_unit_role(unit_name):
    if unit_name in ROLE_OVERRIDES:
        return ROLE_OVERRIDES[unit_name]
    unit = UNIT_LOOKUP.get(unit_name)
    if not unit:
        return "Other"
    return unit["role"] or "Other"

def label_roles(itemized_units):
    roles = {}
    for u in itemized_units:
        role = get_unit_role(u["unit"])
        if role not in roles:
            roles[role] = u["unit"]
    return roles

df["roles"] = df["itemized_units"].apply(label_roles)

ALL_ROLES = ["ADCarry", "ADCaster", "ADReaper", "ADSpecialist", "ADFighter", "ADTank",
             "APCarry", "APCaster", "APReaper", "APFighter", "APTank", "HFighter"]

for role in ALL_ROLES:
    df[role] = df["roles"].apply(lambda r: r.get(role))

df[["match_id", "placement"] + ALL_ROLES].head()

,match_id,placement,ADCarry,ADCaster,ADReaper,ADSpecialist,ADFighter,ADTank,APCarry,APCaster,APReaper,APFighter,APTank,HFighter
0,NA1_5562880650,4,MissFortune,NaN,NaN,NaN,NaN,NaN,NaN,Nami,NaN,Shen,Maokai,NaN
1,NA1_5560691122,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AurelionSol,NaN,Blitzcrank,Nunu,NaN
2,NA1_5559979852,4,MissFortune,NaN,NaN,NaN,NaN,NaN,NaN,Nami,NaN,NaN,Maokai,NaN
3,NA1_5559669327,5,NaN,NaN,NaN,NaN,NaN,NaN,Vex,Karma,NaN,Blitzcrank,Nunu,NaN
4,NA1_5559167035,5,NaN,NaN,NaN,NaN,NaN,NaN,Teemo,Lissandra,NaN,NaN,Leona,NaN


## Carry identification

In [22]:
XP_THRESHOLDS = {2: 2, 3: 6, 4: 10, 5: 20, 6: 36, 7: 56, 8: 80, 9: 84}

def estimate_gold_on_xp(level, last_round):
    xp_needed = XP_THRESHOLDS.get(level, 0)
    auto_xp   = last_round * 2
    return max(0, xp_needed - auto_xp)

def carry_status(itemized_units):
    if not itemized_units:
        return "no_carry", None, None, None
    carry = itemized_units[0]
    unit, cost, tier = carry["unit"], carry["cost"], carry["tier"]
    if cost <= 3:
        status = "hit" if tier >= 3 else "missed"
    else:
        if tier >= 3:
            status = "exceptional"
        elif tier >= 2:
            status = "hit"
        else:
            status = "missed"
    return status, unit, cost, tier

df["gold_on_xp"] = df.apply(lambda r: estimate_gold_on_xp(r["level"], r["last_round"]), axis=1)

carry_cols = df["itemized_units"].apply(
    lambda u: pd.Series(carry_status(u),
                        index=["carry_status", "carry_unit", "carry_cost", "carry_tier"])
)
df = pd.concat([df, carry_cols], axis=1)

df["n_3stars"] = df["itemized_units"].apply(lambda u: sum(1 for x in u if x["tier"] >= 3))
df[["placement", "carry_unit", "carry_cost", "carry_tier", "carry_status", "n_3stars"]].head()

,placement,carry_unit,carry_cost,carry_tier,carry_status,n_3stars
0,4,MissFortune,3.0,3.0,hit,2
1,6,AurelionSol,4.0,2.0,hit,0
2,4,MissFortune,3.0,3.0,hit,1
3,5,Karma,4.0,2.0,hit,0
4,5,Teemo,1.0,3.0,hit,4


## Traits, augments & board aggregates

In [23]:
STYLE_LABELS = {1: "bronze", 2: "silver", 3: "gold", 4: "prismatic"}

def parse_active_traits(traits_list):
    active = []
    for t in traits_list:
        style = t.get("style", 0)
        if style == 0:
            continue
        name = t["name"].replace("TFT17_", "").replace("TFT_", "")
        active.append({"trait": name, "num_units": t["num_units"],
                       "style": STYLE_LABELS.get(style, "active"), "style_level": style})
    active.sort(key=lambda x: (-x["style_level"], -x["num_units"]))
    return active

df["active_traits"]   = df["traits"].apply(parse_active_traits)
df["dominant_trait"]  = df["active_traits"].apply(lambda ts: ts[0]["trait"] if ts else None)
df["top_trait_style"] = df["active_traits"].apply(lambda ts: ts[0]["style"] if ts else None)

df["augment_1"] = df["augments"].apply(lambda a: a[0] if len(a) > 0 else None)
df["augment_2"] = df["augments"].apply(lambda a: a[1] if len(a) > 1 else None)
df["augment_3"] = df["augments"].apply(lambda a: a[2] if len(a) > 2 else None)

df["top4"]             = (df["placement"] <= 4).astype(int)
df["n_2stars"]         = df["itemized_units"].apply(lambda u: sum(1 for x in u if x["tier"] == 2))
df["total_items"]      = df["itemized_units"].apply(lambda u: sum(x["n_items"] for x in u))
df["total_board_cost"] = df["units"].apply(
    lambda units: sum(RARITY_TO_COST.get(u["rarity"], 0)
                      for u in units if u["character_id"].startswith("TFT17_"))
)

all_trait_names = sorted({t["trait"] for ts in df["active_traits"] for t in ts})
for trait in all_trait_names:
    df[f"trait_{trait}"] = df["active_traits"].apply(
        lambda ts: next((t["style_level"] for t in ts if t["trait"] == trait), 0)
    )

print(f"Added {len(all_trait_names)} trait activation columns")
df[["placement", "top4", "dominant_trait", "augment_1", "augment_2", "augment_3",
    "n_2stars", "n_3stars", "total_items", "total_board_cost"]].head()

Added 41 trait activation columns


,placement,top4,dominant_trait,augment_1,augment_2,augment_3,n_2stars,n_3stars,total_items,total_board_cost
0,4,1,APTrait,None,None,None,2,2,13,24
1,6,0,ShieldTank,None,None,None,5,0,13,30
2,4,1,APTrait,None,None,None,3,1,11,23
3,5,0,BlitzcrankUniqueTrait,None,None,None,2,0,10,35
4,5,0,ShenUniqueTrait,None,None,None,1,4,12,16


## Save clean CSV
Drop nested/raw columns — everything important is already flat.

In [24]:
DROP_COLS = ["augments", "traits", "units", "itemized_units", "active_traits", "roles"]

df_clean = (df[(df["level"] > 1) & (df["gold_left"] <= 50)]
              .drop(columns=DROP_COLS)
              .copy())

df_clean.to_csv("data/tft_clean.csv", index=False)

print(f"Saved {len(df_clean):,} rows × {len(df_clean.columns)} columns → data/tft_clean.csv")
print(f"Top-4 rate overall: {df_clean['top4'].mean():.1%}")
print(f"Top-4 rate (me):    {df_clean[df_clean['is_me']]['top4'].mean():.1%}")
print(f"Top-4 rate (chall): {df_clean[~df_clean['is_me']]['top4'].mean():.1%}")
df_clean.head()

Saved 1,556 rows × 84 columns → data/tft_clean.csv
Top-4 rate overall: 63.0%
Top-4 rate (me):    55.4%
Top-4 rate (chall): 63.4%


,match_id,date,player_name,is_me,placement,level,last_round,stage_round,gold_left,damage_to_players,...,trait_Stargazer_Medallion,trait_Stargazer_Mountain,trait_Stargazer_Serpent,trait_Stargazer_Shield,trait_Stargazer_Wolf,trait_SummonTrait,trait_TahmKenchUniqueTrait,trait_Timebreaker,trait_VexUniqueTrait,trait_ZedUniqueTrait
0,NA1_5562880650,2026-05-18 04:53:11.410,me,True,4,9,34,6-5,0,105,...,0,0,0,0,0,0,3,0,0,0
1,NA1_5560691122,2026-05-15 05:37:09.012,me,True,6,9,31,6-2,7,110,...,0,0,0,0,0,1,0,0,0,0
2,NA1_5559979852,2026-05-14 05:08:10.272,me,True,4,9,35,7-1,1,143,...,0,0,0,0,0,0,3,0,0,0
3,NA1_5559669327,2026-05-13 23:18:42.862,me,True,5,9,31,6-2,1,94,...,0,0,0,0,0,1,0,0,3,0
4,NA1_5559167035,2026-05-13 03:01:23.733,me,True,5,8,31,6-2,19,83,...,0,0,0,0,0,1,0,0,0,0
